In [3]:
!pip install -q transformers datasets evaluate rouge_score sentencepiece accelerate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00


In [4]:
import torch
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

import evaluate

In [5]:
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [6]:
dataset = load_dataset("ccdv/pubmed-summarization")

print(dataset)

README.md:   0%|          | 0.00/3.80k [00:00<?, ?B/s]

section/train-00000-of-00005.parquet: reconstructing file:   0%|          |  0.00B /  210MB            

section/train-00000-of-00005.parquet: downloading bytes:           |  0.00B            

section/train-00001-of-00005.parquet: reconstructing file:   0%|          |  0.00B /  208MB            

section/train-00001-of-00005.parquet: downloading bytes:           |  0.00B            

...

section/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/119924 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6633 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6658 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'abstract'],
        num_rows: 119924
    })
    validation: Dataset({
        features: ['article', 'abstract'],
        num_rows: 6633
    })
    test: Dataset({
        features: ['article', 'abstract'],
        num_rows: 6658
    })
})


In [7]:
print("ARTICLE:")
print(dataset["train"][0]["article"][:2000])

print("\nABSTRACT:")
print(dataset["train"][0]["abstract"])

ARTICLE:
a recent systematic analysis showed that in 2011 , 314 ( 296 - 331 ) million children younger than 5 years were mildly , moderately or severely stunted and 258 ( 240 - 274 ) million were mildly , moderately or severely underweight in the developing countries . 
 in iran a study among 752 high school girls in sistan and baluchestan showed prevalence of 16.2% , 8.6% and 1.5% , for underweight , overweight and obesity , respectively . 
 the prevalence of malnutrition among elementary school aged children in tehran varied from 6% to 16% . 
 anthropometric study of elementary school students in shiraz revealed that 16% of them suffer from malnutrition and low body weight . 
 snack should have 300 - 400 kcal energy and could provide 5 - 10 g of protein / day . nowadays , school nutrition programs are running as the national programs , world - wide . national school lunch program in the united states 
 there are also some reports regarding school feeding programs in developing countr

In [8]:
train_dataset = dataset["train"].select(range(5000))
validation_dataset = dataset["validation"].select(range(500))
test_dataset = dataset["test"].select(range(500))

print(len(train_dataset))
print(len(validation_dataset))
print(len(test_dataset))

5000
500
500


In [9]:
model_name = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [10]:
prefix = "summarize: "

def preprocess_function(examples):

    inputs = [
        prefix + text
        for text in examples["article"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["abstract"],
        max_length=128,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [11]:
tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_validation = validation_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=validation_dataset.column_names
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names
)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [12]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [13]:
rouge = evaluate.load("rouge")

In [18]:
def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    # Some Trainer versions return a tuple
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Convert predictions to numpy
    predictions = np.asarray(predictions)

    # Make sure token IDs are valid integers
    predictions = predictions.astype(np.int64)

    # Replace invalid label padding values (-100)
    labels = np.where(
        labels != -100,
        labels,
        tokenizer.pad_token_id
    )

    labels = np.asarray(labels).astype(np.int64)

    # Decode predictions
    decoded_predictions = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    # Decode reference summaries
    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    # Remove extra whitespace
    decoded_predictions = [
        pred.strip()
        for pred in decoded_predictions
    ]

    decoded_labels = [
        label.strip()
        for label in decoded_labels
    ]

    result = rouge.compute(
        predictions=decoded_predictions,
        references=decoded_labels
    )

    return {
        "rouge1": result["rouge1"],
        "rouge2": result["rouge2"],
        "rougeL": result["rougeL"]
    }

In [25]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./medical_summarizer",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    weight_decay=0.01,

    num_train_epochs=1,

    # IMPORTANT
    predict_with_generate=False,

    fp16=torch.cuda.is_available(),

    logging_steps=100,

    save_total_limit=2,

    report_to="none"
)

In [26]:
generation_max_length=128
generation_num_beams=4

In [27]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=None
)

In [28]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.818992,2.529457


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=2.7836473388671874, metrics={'train_runtime': 174.7935, 'train_samples_per_second': 28.605, 'train_steps_per_second': 7.151, 'total_flos': 676709007360000.0, 'train_loss': 2.7836473388671874, 'epoch': 1.0})

In [29]:
results = trainer.evaluate()

print(results)

Training Loss,Validation Loss,Epoch
2.818992,2.529457,1


{'eval_loss': 2.529456615447998}


In [30]:
def summarize_medical_report(text):

    inputs = tokenizer(
        "summarize: " + text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    summary_ids = model.generate(
        **inputs,
        max_new_tokens=100,
        min_new_tokens=20,
        num_beams=4,
        early_stopping=True
    )

    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary

In [31]:
sample_report = """
A 56-year-old patient presented with persistent cough,
fever and shortness of breath for seven days.
Chest imaging demonstrated bilateral pulmonary infiltrates.
Laboratory investigations showed elevated inflammatory markers.
The patient was diagnosed with a respiratory infection
and was started on appropriate antimicrobial treatment.
"""

summary = summarize_medical_report(sample_report)

print("MEDICAL REPORT:")
print(sample_report)

print("\nAI GENERATED SUMMARY:")
print(summary)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=20) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MEDICAL REPORT:

A 56-year-old patient presented with persistent cough,
fever and shortness of breath for seven days.
Chest imaging demonstrated bilateral pulmonary infiltrates.
Laboratory investigations showed elevated inflammatory markers.
The patient was diagnosed with a respiratory infection
and was started on appropriate antimicrobial treatment.


AI GENERATED SUMMARY:
a 56-year-old patient presented with persistent cough, fever and shortness of breath for seven days. the patient was diagnosed with a respiratory infection and was started on appropriate antimicrobial treatment.


In [32]:
import evaluate

rouge = evaluate.load("rouge")

In [34]:
predictions = []
references = []

# Evaluate only 50 reports first
num_samples = 50

for i in range(num_samples):

    report = test_dataset[i]["article"]
    reference = test_dataset[i]["abstract"]

    generated = summarize_medical_report(report)

    predictions.append(generated)
    references.append(reference)

    if i < 3:
        print("=" * 80)
        print("GENERATED:")
        print(generated)

        print("\nREFERENCE:")
        print(reference)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=20) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=20) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

GENERATED:
anxiety affects quality of life in those living with parkinson's disease ( pd ) more so than overall cognitive status, motor deficits, apathy, and depression. however, our current understanding of anxiety and its impact on cognition in pd remains meager and lags far behind that of depression. however, few studies have specifically investigated the relationship between anxiety and cognition in pd. however 

REFERENCE:
research on the implications of anxiety in parkinson 's disease ( pd ) has been neglected despite its prevalence in nearly 50% of patients and its negative impact on quality of life . 
 previous reports have noted that neuropsychiatric symptoms impair cognitive performance in pd patients ; however , to date , no study has directly compared pd patients with and without anxiety to examine the impact of anxiety on cognitive impairments in pd . 
 this study compared cognitive performance across 50 pd participants with and without anxiety ( 17 pda+ ; 33 pda ) , who u

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=20) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GENERATED:
small non - coding rnas are transcribed into mrna but remain untranslated in eukaryotic cells. they include sirna ( small interfering rna ), mirna ( microrna ), pirna ( piwi - interacting rna ) and snorna ( small nucleolar rna

REFERENCE:
small non - coding rnas include sirna , mirna , pirna and snorna . 
 the involvement of mirnas in the regulation of mammary gland tumorigenesis has been widely studied while the role for other small non - coding rnas remains unclear . here 
 we summarize the involvement of mirna in breast cancer onset and progression through regulating the cell cycle and cellular proliferation . 
 the regulation of breast cancer stem cells and tumor regeneration by mirna is reviewed . 
 in addition , the emerging evidence demonstrating the involvement of pirna and snorna in breast cancer is briefly described .


[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=20) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GENERATED:
ohss is a serious complication of ovulation induction, occurring in 1 - 10% of in vitro fertilization patients. this iatrogenic condition has a spectrum of clinical and laboratory manifestations ranging from mild to severe, even life - threatening conditions. among the serious manifestations of ohss are ascites and pleural effusion ( rizk and smitz

REFERENCE:
objective : to evaluate the efficacy and safety of outpatient management of severe ovarian hyperstimulation syndrome  ( ohss ) requiring placement of a pigtail catheter.methods : retrospective analysis of thirty - three consecutive patients who underwent in - vitro fertilization  ( 2003 - 2009 ) and developed severe / critical ohss requiring placement of a pigtail catheter . 
 patients who were managed on outpatient basis were monitored by frequent office visits , daily phone calls , and received iv normal saline for hydration when required.results : in 3 patients  ( 9.1% ) ohss started early , requiring placement of a

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=20) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=20) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

In [35]:
results = rouge.compute(
    predictions=predictions,
    references=references
)

print(results)

{'rouge1': np.float64(0.24247039983142038), 'rouge2': np.float64(0.09206051385343039), 'rougeL': np.float64(0.17052384015507827), 'rougeLsum': np.float64(0.2115147767019254)}


In [36]:
!pip install -q pypdf pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 9.3 MB/s eta 0:00:00


In [37]:
import pandas as pd
from pypdf import PdfReader
from google.colab import files

In [38]:
def extract_pdf_text(pdf_path):

    reader = PdfReader(pdf_path)

    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text

In [39]:
def extract_pdf_text(pdf_path):

    reader = PdfReader(pdf_path)

    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text

In [40]:
uploaded = files.upload()

filename = list(uploaded.keys())[0]

print("Uploaded file:", filename)

Saving Sample-filled-in-MR.pdf to Sample-filled-in-MR.pdf
Uploaded file: Sample-filled-in-MR.pdf


In [41]:
if filename.lower().endswith(".csv"):

    df = pd.read_csv(filename)

    print("CSV loaded successfully!")
    print("Columns:", df.columns.tolist())

    display(df.head())

elif filename.lower().endswith(".pdf"):

    medical_report = extract_pdf_text(filename)

    print("PDF loaded successfully!")
    print("\nExtracted text:")
    print(medical_report[:3000])

else:

    print("Please upload a CSV or PDF file.")

PDF loaded successfully!

Extracted text:
- 1 - 
 
SAMPLE 
(All names and details provided in this sample are fictitious.  
Some fields have been deliberately left blank.) 
 
 
MEDICAL REPORT 
 
SECTION 1: PATIENT’S PARTICULARS 
 
Full name of patient: Mr Tan Ah Kow 
 
NRIC/FIN/Passport no. of patient: S1111111X 
 
Age of patient: 55 years old 
 
SECTION 2: DOCTOR’S PARTICULARS 
 
Full name of doctor: Tan Ah Moi 
 
NRIC/FIN/Passport no. of doctor: S2222222Z  
 
 
MCR no. of doctor: 333333 
 
Hospital / Clinic name and address: 1 Blackacre Hospital, Singapore 01010101 
 
 
 
Doctor’s qualifications and experience in this area of work: 
 
 
[To set out details] 
 
 
 
 
 
 
 
- 2 - 
 
 
Doctor-patient relationship: 
 
Please state if you have been seeing the patient regularly over a period of time (if so, 
please state when you first started see ing the patient and how often you see the 
patient) or if you saw the patient specifically for this mental capacity assessment only. 
 
 
I have

In [42]:
if filename.lower().endswith(".pdf"):

    summary = summarize_medical_report(medical_report)

    print("=" * 80)
    print("AI GENERATED MEDICAL SUMMARY")
    print("=" * 80)

    print(summary)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=20) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AI GENERATED MEDICAL SUMMARY
MEDICAL REPORT SECTION 1: PATIENT’S PARTICULARS Full name of patient: Tan Ah Moi NRIC/FIN/Passport no. of patient: S1111111X Age of patient: 55 years old SECTION 2: DOCTOR’s PARTICULARS Full name of doctor: Tan Ah Moi NRIC/FIN/Passport no. of doctor: 333333 Hospital / Clinic name and address: 1 Blackacre Hospital
